# DRISHTI — RELLIS-3D download: Velodyne Ultra Puck (32-ch) stream

Ticket #3 (⬜ DETACHABLE — runs for hours, blocks nothing else; kick it off and keep working on other tickets).

This is the **optional, lighter** stream — 32-channel Velodyne Ultra Puck. Build Map Ticket #3 marks it "optional, not required": the build targets the 64-channel Ouster OS1 primarily (`colab_download_rellis_os1.ipynb`), but this stream is useful on its own as a second real 32-beam sensor for the Ticket #59 portability ablation (alongside nuScenes' HDL-32E), or simply because 5.58GB is a much smaller/faster download than 14GB.

**Standalone notebook** — does not require `colab_setup.ipynb` or the OS1 notebook to have run first, but does need the repo, which this notebook clones if missing.

**Size/time reality check**: the archive is 5.58GB + 143.6MB labels (~5.72GB transient download) across all 5 sequences combined; only `00004/` is kept. **Unlike the Ouster stream, RELLIS-3D does not publish a separate poses file for the Velodyne stream** — `perception/rellis_loader.py` already handles this gracefully (falls back to an identity pose when `poses.txt` is absent), but be aware `T_world` will not be meaningful for this stream unless you supply poses some other way.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone once per session; re-running is safe (pulls instead of re-cloning).
import os
REPO_URL = 'REPLACE_WITH_YOUR_GITHUB_URL'  # push this repo to GitHub first, per Ticket #1 "Watch out"
if not os.path.isdir('drishti'):
    !git clone {REPO_URL} drishti
%cd drishti
!git pull

In [ ]:
!pip install -q gdown

## The download

Downloads the 5.58GB Velodyne SemanticKITTI archive + 143.6MB labels to `/content/drive/MyDrive/drishti_data/rellis_raw/`, extracts only `00004/`, and copies the result to `/content/drive/MyDrive/drishti_data/rellis/00004/` — the same sequence folder the OS1 notebook uses, so both streams can coexist side by side there.

In [ ]:
DEST_ROOT = '/content/drive/MyDrive/drishti_data'
!bash scripts/download_rellis.sh {DEST_ROOT} vel

## Verify

Confirm the extraction landed where the loader expects it. Note this does **not** run `tests/test_rellis_loader.py::test_real_sequence_00004` -- that test hardcodes `sensor_stream='os1'` (no parameter to select the stream), so it isn't meaningful against a Velodyne-only download. This calls the loader directly with `sensor_stream='vel'` instead, which is the actually-correct way to exercise this stream.

In [ ]:
import os
seq_dir = f'{DEST_ROOT}/rellis/00004'
for sub in ('vel_cloud_node_kitti_bin', 'vel_cloud_node_semantickitti_label_id'):
    p = os.path.join(seq_dir, sub)
    print(p, '->', 'OK' if os.path.exists(p) else 'MISSING')
!ls -la {seq_dir} 2>/dev/null | head -20

In [ ]:
from perception.rellis_loader import load_rellis_sweep

sweep = load_rellis_sweep(seq_dir, frame_idx=0, sensor_stream='vel')
print('sensor_id:', sweep.sensor_id)
print('points:', sweep.xyz.shape)
print('intensity range:', sweep.intensity.min(), sweep.intensity.max())
assert sweep.xyz.shape[0] > 0, 'loaded zero points -- check the extraction above'
print('OK -- frame 0 of the Velodyne stream loads.')

## Cleanup (optional, once the above passes)

`rellis_raw/` holds the full transient archives (5.58GB+) — you only need `rellis/00004/` going forward. Delete `rellis_raw/` to free Drive space; do **not** delete `rellis/00004/`. If you also ran the OS1 notebook, both streams' output live in the same `rellis/00004/` folder and this only clears the shared `rellis_raw/` staging area.

In [ ]:
# Uncomment once you've verified the cells above passed:
# import shutil
# shutil.rmtree(f'{DEST_ROOT}/rellis_raw', ignore_errors=True)